# Pipeline demo: MegaDetector v6 → BioCLIP-2

Minimal end-to-end run on a small image folder. Same code path as the `wytrap detect` CLI — use this notebook for exploration and the CLI for SLURM jobs.

**Setup**: install wytrap once into your environment:

```bash
pip install -e ../wytrap
```

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from wytrap.run import process_folder
from wytrap.io import load_record

In [ ]:
# Edit these two paths.
INPUT_DIR  = "/path/to/sample_images"
OUTPUT_DIR = "/path/to/sample_results"

summary = process_folder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    species="wyoming_all",   # or 'wyoming_mammals', 'ynp_testbed', or a custom list
    device="auto",
    recursive=False,
    resume=True,
)
summary

## Render boxes + labels for the first few results

In [ ]:
result_files = sorted(Path(OUTPUT_DIR).glob("*.json"))[:6]

for jf in result_files:
    rec = load_record(jf)
    if rec.error:
        print(f"{jf.name}: error {rec.error}")
        continue
    img = Image.open(rec.image_path).convert("RGB")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)
    ax.set_title(jf.stem)
    for det in rec.detections:
        x1, y1, x2, y2 = det.box_xyxy
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            edgecolor="yellow", facecolor="none", linewidth=2,
        ))
        ax.text(x1, max(0, y1 - 6),
                f"{det.label} ({det.cls_score:.2f})",
                color="yellow", fontsize=10,
                bbox=dict(boxstyle="round", fc="black", ec="none", alpha=0.6))
    ax.axis("off")
    plt.show()

## CLI equivalent

From a shell on a GPU node:

```bash
wytrap detect \
    --input  /path/to/sample_images \
    --output /path/to/sample_results \
    --species wyoming_all \
    --device cuda --resume
```